In [2]:
import pandas as pd
import numpy as np

def prepare_hospital_data(patients_path, services_path, schedule_path):
    # 1. Load Data
    patients_df = pd.read_csv(patients_path)
    services_df = pd.read_csv(services_path)
    schedule_df = pd.read_csv(schedule_path)
    
    # 2. Date Processing & Length of Stay 
    patients_df['arrival_date'] = pd.to_datetime(patients_df['arrival_date'])
    patients_df['departure_date'] = pd.to_datetime(patients_df['departure_date'])
    patients_df['length_of_stay'] = (patients_df['departure_date'] - patients_df['arrival_date']).dt.days
    
    # 3. Aggregate Patient Data to Weekly level
    min_date = patients_df['arrival_date'].min()
    patients_df['week'] = ((patients_df['arrival_date'] - min_date).dt.days // 7) + 1
    weekly_los = patients_df.groupby(['week', 'service'])['length_of_stay'].mean().reset_index()
    weekly_los.rename(columns={'length_of_stay': 'avg_los'}, inplace=True)
    
    # 4. Aggregate Staffing to Weekly level
    weekly_staff = schedule_df.groupby(['week', 'service'])['present'].sum().reset_index()
    weekly_staff.rename(columns={'present': 'staff_on_duty'}, inplace=True)
    
    # 5. The Master Merge (Fix: Use services_df, not s_df)
    df = pd.merge(services_df, weekly_staff, on=['week', 'service'], how='left')
    df = pd.merge(df, weekly_los, on=['week', 'service'], how='left')
    
    # 6. Target & Feature Engineering
    df['utilization_rate'] = (df['patients_admitted'] / df['available_beds']).clip(upper=1.0)
    df['demand_pressure'] = df['patients_request'] / df['available_beds']
    
    # 7. Time-Series Lags
    df = df.sort_values(['service', 'week'])
    df['utilization_lag1'] = df.groupby('service')['utilization_rate'].shift(1)
    
    # 8. Handling Nulls
    df['avg_los'] = df.groupby('service')['avg_los'].transform(lambda x: x.fillna(x.median()))
    df = df.dropna(subset=['utilization_lag1'])
    
    return df

